In [1]:
from cmdstanpy import CmdStanModel
import numpy as np
import pandas as pd
from pathlib import Path
import scipy.stats as st
import os

from joblib import Parallel, delayed
from tqdm.auto import tqdm
import traceback

from __future__ import annotations
from dataclasses import dataclass
from typing import Any, Dict, Optional

#### Compile Stan model

In [5]:
stan_path = Path("/Users/katsiarynadavydzenka/Documents/PhD_AI/DeConveil/stan/differential_dosage_model.stan")
model = CmdStanModel(stan_file=str(stan_path))

#### Fit one gene / multiple genes

In [7]:
DATA_PATH = "/Users/katsiarynadavydzenka/Documents/PhD_AI/CRC_case_study/data/stan_model_test/"
df_long = pd.read_csv(os.path.join(DATA_PATH, "crc_joint_long_de.csv"))
df_long.head()

,gene,sampleID,expr,copies,subtype,purity,stroma,sf,eup_dev_cancer,eup_equiv_cancer,eup_equiv_centered
0,A1BG,CRC.SW.U0001.T,12,3.0,MSS,0.41,0.59,1.293201,0.5,1.5,0.5
1,A1BG,CRC.SW.U0002.T,3,2.0,MSS,0.37,0.63,1.024008,0.0,1.0,0.0
2,A1BG,CRC.SW.U0004.T,14,3.0,MSS,0.58,0.42,1.169870,0.5,1.5,0.5
3,A1BG,CRC.SW.U0030.T,4,2.0,MSS,0.59,0.41,1.290047,0.0,1.0,0.0
4,A1BG,CRC.SW.U0066.T,4,2.0,MSS,0.41,0.59,0.971163,0.0,1.0,0.0


In [9]:
# Select one gene
gene_df = df_long[df_long["gene"] == "SMAD4"]
gene_df.head()

,gene,sampleID,expr,copies,subtype,purity,stroma,sf,eup_dev_cancer,eup_equiv_cancer,eup_equiv_centered
13578206,SMAD4,CRC.SW.U0001.T,2367,2.0,MSS,0.41,0.59,1.293201,0.0,1.0,0.0
13578207,SMAD4,CRC.SW.U0002.T,1990,2.0,MSS,0.37,0.63,1.024008,0.0,1.0,0.0
13578208,SMAD4,CRC.SW.U0004.T,2449,1.0,MSS,0.58,0.42,1.169870,-0.5,0.5,-0.5
13578209,SMAD4,CRC.SW.U0030.T,2976,2.0,MSS,0.59,0.41,1.290047,0.0,1.0,0.0
13578210,SMAD4,CRC.SW.U0066.T,1617,2.0,MSS,0.41,0.59,0.971163,0.0,1.0,0.0


In [7]:
# Select multiple genes
gene_df = df_long[df_long["gene"].isin(["BRAF", "KRAS", "PIK3CA", "APC", "SMAD4"])]
gene_df.head()

,gene,sampleID,expr,copies,subtype,purity,stroma,sf,eup_dev_cancer,eup_equiv_cancer,eup_equiv_centered
783870,APC,CRC.SW.U0001.T,4685,2.0,MSS,0.41,0.59,1.293201,0.0,1.0,0.0
783871,APC,CRC.SW.U0002.T,2625,2.0,MSS,0.37,0.63,1.024008,0.0,1.0,0.0
783872,APC,CRC.SW.U0004.T,2069,2.0,MSS,0.58,0.42,1.169870,0.0,1.0,0.0
783873,APC,CRC.SW.U0030.T,3568,2.0,MSS,0.59,0.41,1.290047,0.0,1.0,0.0
783874,APC,CRC.SW.U0066.T,2646,2.0,MSS,0.41,0.59,0.971163,0.0,1.0,0.0


In [9]:
def fit_one_gene_de(
    gene_df: pd.DataFrame,
    model: "CmdStanModel",
    gene: str | None = None,
    cna: str = "all",
    et: float = 0.15,
    min_aneup: int = 5,
    min_unique_counts: int = 5,
    min_cn_abs_sum: float = 1.0,
    subtype_col: str = "subtype",
    subtype_order: list[str] | None = None,
    chains: int = 4,
    iter_warmup: int = 500,
    iter_sampling: int = 500,
    seed: int = 1,
    show_progress: bool = False,
    adapt_delta: float = 0.9,
    max_treedepth: int = 12,
    rope_logfc: float = float(np.log(1.2)),  # ROPE for tumor DE (natural log)
    eps_frac: float = 0.10,                  # ROPE for fractional CN effect (e.g. 10%)
    eps_lp: float = float(np.log(1.1)),      # ROPE on log-effect pieces (~10%)
    return_all_subtypes: bool = True,
):
    """
    Log-link subtype-aware CN-expression NB model (identified mechanistic version).

    Required columns:
      gene, expr, copies, purity, sf, subtype_col

    Expects Stan outputs:
      - contrasts: delta_tumor0_log, delta_scaling, delta_dev
      - dispersion: phi
      - per-subtype (optional but recommended): b0[s], b_scaling[s], b_deviation[s]
      - GQs:
          lp_2to1[s], lp_2to3[s], lp_2to4[s]
          lp_scaling_2to1[s], lp_dev_2to1[s]
          lp_scaling_2to3[s], lp_dev_2to3[s]
          lp_scaling_2to4[s], lp_dev_2to4[s]
          (optional) cancel_index_2to{1,3,4}[s]
    """

    df = gene_df.copy()

    # subset gene
    if gene is not None and "gene" in df.columns and df["gene"].nunique() > 1:
        df = df.loc[df["gene"] == gene].copy()
    if gene is None and "gene" in df.columns and df["gene"].nunique() == 1:
        gene = str(df["gene"].iloc[0])

    if df.empty:
        return {"status": "skipped", "gene": gene, "reason": "no_rows_for_gene"}

    required = {"expr", "copies", "purity", "sf", subtype_col}
    missing = required - set(df.columns)
    if missing:
        return {"status": "error", "gene": gene, "reason": f"missing_columns: {sorted(missing)}"}

    # CNA subset
    if cna == "amp":
        df = df[df["copies"] > (2 - et)]
    elif cna == "del":
        df = df[df["copies"] < (2 + et)]
    elif cna == "all":
        pass
    else:
        raise ValueError("cna must be 'amp', 'del', or 'all'")

    if df.empty:
        return {"status": "skipped", "gene": gene, "reason": "no_samples_after_cna_filter"}

    # QC
    df = df.dropna(subset=list(required))
    if df.empty:
        return {"status": "skipped", "gene": gene, "reason": "all_na_after_dropna"}

    if (df["expr"] < 0).any():
        return {"status": "error", "gene": gene, "reason": "negative_counts"}

    if not df["purity"].between(0, 1).all():
        return {"status": "error", "gene": gene, "reason": "purity_out_of_bounds"}

    if not (df["sf"] > 0).all():
        return {"status": "error", "gene": gene, "reason": "nonpositive_sf"}

    if df["expr"].nunique() < min_unique_counts:
        return {"status": "skipped", "gene": gene, "reason": "too_few_unique_counts"}

    # aneuploid check
    n_aneup = int((np.abs(df["copies"].astype(float) - 2.0) > (1.0 - et)).sum())
    if n_aneup < min_aneup or (df["expr"] == 0).all():
        return {"status": "skipped", "gene": gene, "n_aneup": n_aneup, "reason": "low_aneup_or_all_zero"}

    # identifiability check: need some CN deviation mass
    dev_tmp = (df["copies"].astype(float) - 2.0) / 2.0
    if cna == "all" and float(np.abs(dev_tmp).sum()) < min_cn_abs_sum:
        return {"status": "skipped", "gene": gene, "n_aneup": n_aneup, "reason": "too_little_cn_variation"}

    # subtype encoding
    if subtype_order is not None:
        cat = pd.Categorical(df[subtype_col], categories=subtype_order, ordered=True)
        if cat.isna().any():
            bad = df.loc[cat.isna(), subtype_col].unique().tolist()
            return {"status": "error", "gene": gene, "reason": f"unknown_subtypes: {bad}"}
        subtype_codes = (cat.codes + 1).astype(int)
        levels = list(cat.categories)
    else:
        levels = sorted(pd.unique(df[subtype_col]).tolist())
        mapping = {lv: i + 1 for i, lv in enumerate(levels)}
        subtype_codes = df[subtype_col].map(mapping).astype(int).to_numpy()

    S = len(levels)
    if S < 2:
        return {"status": "skipped", "gene": gene, "reason": "need_at_least_2_subtypes_present_for_DE"}

    # CN covariates (identified)
    df["dose_log"] = np.log(np.maximum(df["copies"].astype(float), 0.1) / 2.0)
    df["dev"] = (df["copies"].astype(float) - 2.0) / 2.0

    stan_data = {
        "N": int(len(df)),
        "y": df["expr"].astype(int).to_numpy(),
        "S": int(S),
        "subtype": np.asarray(subtype_codes, dtype=int),
        "sf": df["sf"].to_numpy(dtype=float),
        "purity": df["purity"].to_numpy(dtype=float),
        "dose_log": df["dose_log"].to_numpy(dtype=float),
        "dev": df["dev"].to_numpy(dtype=float),
    }

    rng = np.random.default_rng(seed)
    init = {
        "b0_mean": float(rng.normal(0.0, 0.2)),
        "b0_offset": rng.normal(0.0, 0.1, size=S).tolist(),
        "b_scaling_mean": float(rng.normal(0.0, 0.2)),
        "b_scaling_offset": rng.normal(0.0, 0.1, size=S).tolist(),
        "b_dev_mean": float(rng.normal(0.0, 0.05)),
        "b_dev_offset": rng.normal(0.0, 0.05, size=S).tolist(),
        "b_noncancer_log": float(rng.normal(0.0, 0.2)),
        "phi": 1.0,
    }

    fit = model.sample(
        data=stan_data,
        chains=chains,
        iter_warmup=iter_warmup,
        iter_sampling=iter_sampling,
        seed=seed,
        inits=init,
        show_progress=show_progress,
        adapt_delta=adapt_delta,
        max_treedepth=max_treedepth,
    )

    draws = fit.draws_pd()

    # ---- helpers ----
    def q(x):
        return np.quantile(x, [0.025, 0.5, 0.975])

    def summarize_draw(x, prefix):
        qi = q(x)
        return {
            f"{prefix}_mean": float(np.mean(x)),
            f"{prefix}_q025": float(qi[0]),
            f"{prefix}_q50":  float(qi[1]),
            f"{prefix}_q975": float(qi[2]),
        }

    # check essential columns
    required_cols = ["delta_tumor0_log", "delta_scaling", "delta_dev", "phi"]
    missing_cols = [c for c in required_cols if c not in draws.columns]
    if missing_cols:
        return {"status": "error", "gene": gene, "reason": f"missing_draws_columns: {missing_cols}"}

    # ---- contrasts ----
    d_tumor = draws["delta_tumor0_log"].to_numpy()
    d_scal  = draws["delta_scaling"].to_numpy()
    d_dev   = draws["delta_dev"].to_numpy()

    # tumor DE: report log2FC
    ln2 = np.log(2.0)
    lfc_tumor = d_tumor / ln2
    lfc_ci = q(lfc_tumor)

    p_up_tumor = float((d_tumor > 0).mean())
    lfsr_tumor = float(min(p_up_tumor, 1 - p_up_tumor))
    p_rope_tumor = float((np.abs(d_tumor) <= rope_logfc).mean())

    p_up_scal = float((d_scal > 0).mean())
    lfsr_scal = float(min(p_up_scal, 1 - p_up_scal))

    p_up_dev = float((d_dev > 0).mean())
    lfsr_dev = float(min(p_up_dev, 1 - p_up_dev))

    out = {
        "status": "ok",
        "gene": gene,
        "N": int(len(df)),
        "n_aneup": n_aneup,
        "cna": cna,
        "subtype_levels": levels,

        # tumor baseline DE at diploid CN (log2FC)
        "tumor0_lfc_mean": float(np.mean(lfc_tumor)),
        "tumor0_lfc_q025": float(lfc_ci[0]),
        "tumor0_lfc_q975": float(lfc_ci[2]),
        "p_up_tumor": p_up_tumor,
        "lfsr_tumor": lfsr_tumor,
        "p_rope_tumor": p_rope_tumor,

        # differential wiring (natural-log parameter scale)
        "delta_scaling_mean": float(np.mean(d_scal)),
        "delta_scaling_q025": float(q(d_scal)[0]),
        "delta_scaling_q975": float(q(d_scal)[2]),
        "p_up_scaling": p_up_scal,
        "lfsr_scaling": lfsr_scal,

        "delta_dev_mean": float(np.mean(d_dev)),
        "delta_dev_q025": float(q(d_dev)[0]),
        "delta_dev_q975": float(q(d_dev)[2]),
        "p_up_dev": p_up_dev,
        "lfsr_dev": lfsr_dev,

        # dispersion
        "phi_mean": float(np.mean(draws["phi"].to_numpy())),
        "phi_q025": float(q(draws["phi"].to_numpy())[0]),
        "phi_q975": float(q(draws["phi"].to_numpy())[2]),
    }

    # ---- subtype-specific coefficients + mechanistic dosage summaries ----
    s_iter = range(1, S + 1) if return_all_subtypes else range(1, min(S, 2) + 1)

    # steps we summarize
    steps = ["2to1", "2to3", "2to4"]

    for s in s_iter:
        # coefficients if present
        for base in ["b0", "b_scaling", "b_deviation"]:
            col = f"{base}[{s}]"
            if col in draws.columns:
                out.update(summarize_draw(draws[col].to_numpy(), f"{base}_s{s}"))
            else:
                out[f"{base}_s{s}_missing"] = True

        for step in steps:
            # net lp + frac
            col_lp = f"lp_{step}[{s}]"
            if col_lp in draws.columns:
                lp = draws[col_lp].to_numpy()
                out.update(summarize_draw(lp, f"lp_{step}_s{s}"))

                frac = np.expm1(lp)
                out.update(summarize_draw(frac, f"fracCN_{step}_s{s}"))

                out[f"p_fracCN_{step}_pos_s{s}"]  = float((frac > eps_frac).mean())
                out[f"p_fracCN_{step}_rope_s{s}"] = float((np.abs(frac) <= eps_frac).mean())
                out[f"p_fracCN_{step}_neg_s{s}"]  = float((frac < -eps_frac).mean())

            # mechanistic pieces: scaling-only and dev-only
            col_sc = f"lp_scaling_{step}[{s}]"
            if col_sc in draws.columns:
                lp_sc = draws[col_sc].to_numpy()
                out.update(summarize_draw(lp_sc, f"lp_scaling_{step}_s{s}"))
                out[f"p_lp_scaling_{step}_pos_s{s}"]  = float((lp_sc >  eps_lp).mean())
                out[f"p_lp_scaling_{step}_rope_s{s}"] = float((np.abs(lp_sc) <= eps_lp).mean())
                out[f"p_lp_scaling_{step}_neg_s{s}"]  = float((lp_sc < -eps_lp).mean())

            col_de = f"lp_dev_{step}[{s}]"
            if col_de in draws.columns:
                lp_de = draws[col_de].to_numpy()
                out.update(summarize_draw(lp_de, f"lp_dev_{step}_s{s}"))
                out[f"p_lp_dev_{step}_pos_s{s}"]  = float((lp_de >  eps_lp).mean())
                out[f"p_lp_dev_{step}_rope_s{s}"] = float((np.abs(lp_de) <= eps_lp).mean())
                out[f"p_lp_dev_{step}_neg_s{s}"]  = float((lp_de < -eps_lp).mean())

            # cancellation index (optional)
            col_ci = f"cancel_index_{step}[{s}]"
            if col_ci in draws.columns:
                out.update(summarize_draw(draws[col_ci].to_numpy(), f"cancel_index_{step}_s{s}"))

        # ---- DC probabilities (gain and loss) for subtype s ----
        # DC_gain: net ~0 for 2->3 AND scaling positive AND dev negative
        if (f"lp_2to3[{s}]" in draws.columns and
            f"lp_scaling_2to3[{s}]" in draws.columns and
            f"lp_dev_2to3[{s}]" in draws.columns):
            lp_net = draws[f"lp_2to3[{s}]"].to_numpy()
            frac_net = np.expm1(lp_net)
            lp_sc = draws[f"lp_scaling_2to3[{s}]"].to_numpy()
            lp_de = draws[f"lp_dev_2to3[{s}]"].to_numpy()

            dc_gain = (np.abs(frac_net) <= eps_frac) & (lp_sc > eps_lp) & (lp_de < -eps_lp)
            out[f"p_DC_gain_s{s}"] = float(dc_gain.mean())

        # DC_loss: net ~0 for 2->1 AND scaling negative AND dev positive
        if (f"lp_2to1[{s}]" in draws.columns and
            f"lp_scaling_2to1[{s}]" in draws.columns and
            f"lp_dev_2to1[{s}]" in draws.columns):
            lp_net = draws[f"lp_2to1[{s}]"].to_numpy()
            frac_net = np.expm1(lp_net)
            lp_sc = draws[f"lp_scaling_2to1[{s}]"].to_numpy()
            lp_de = draws[f"lp_dev_2to1[{s}]"].to_numpy()

            dc_loss = (np.abs(frac_net) <= eps_frac) & (lp_sc < -eps_lp) & (lp_de > eps_lp)
            out[f"p_DC_loss_s{s}"] = float(dc_loss.mean())

    # ---- diagnostics ----
    summ = fit.summary()
    rhat_col = next((c for c in ["R_hat", "Rhat"] if c in summ.columns), None)
    ess_col  = next((c for c in ["Ess_bulk", "ESS_bulk", "N_Eff", "Ess"] if c in summ.columns), None)

    if rhat_col and "phi" in summ.index:
        out["Rhat_phi"] = float(summ.loc["phi", rhat_col])
    if ess_col and "phi" in summ.index:
        out["ess_phi"] = float(summ.loc["phi", ess_col])

    out["fit_flag"] = "ok"
    if ("Rhat_phi" in out and out["Rhat_phi"] > 1.05) or ("ess_phi" in out and out["ess_phi"] < 200):
        out["fit_flag"] = "warn"

    return out

In [11]:
# more diagnostic parameters

def fit_one_gene_de(
    gene_df: pd.DataFrame,
    model: "CmdStanModel",
    gene: str | None = None,
    cna: str = "all",
    et: float = 0.15,
    min_aneup: int = 5,
    min_unique_counts: int = 5,
    min_cn_abs_sum: float = 1.0,
    subtype_col: str = "subtype",
    subtype_order: list[str] | None = None,
    chains: int = 4,
    iter_warmup: int = 500,
    iter_sampling: int = 500,
    seed: int = 1,
    show_progress: bool = False,
    adapt_delta: float = 0.9,
    max_treedepth: int = 12,
    rope_logfc: float = float(np.log(1.2)),
    eps_frac: float = 0.10,          # ROPE on fracCN for ~10% change
    return_all_subtypes: bool = True,
):
    """
    Fit the Bayesian differential gene-dosage model for a single gene.

    Required columns in gene_df:
        expr, copies, purity, sf, subtype_col
    Optional:
        gene (if gene_df contains multiple genes)

    Stan model is the log-link CN-expression model with generated quantities:
        delta_tumor0_log, delta_scaling, delta_dev,
        lp_2to1[s], lp_2to3[s], lp_2to4[s],
        (and optionally fracCN_*, cancel_index_*, p_DC_* if you added them).
    """

    df = gene_df.copy()

    # ---- subset gene ----
    if gene is not None and "gene" in df.columns and df["gene"].nunique() > 1:
        df = df.loc[df["gene"] == gene].copy()
    if gene is None and "gene" in df.columns and df["gene"].nunique() == 1:
        gene = str(df["gene"].iloc[0])

    if df.empty:
        return {"status": "skipped", "gene": gene, "reason": "no_rows_for_gene"}

    required = {"expr", "copies", "purity", "sf", subtype_col}
    missing = required - set(df.columns)
    if missing:
        return {"status": "error", "gene": gene, "reason": f"missing_columns: {sorted(missing)}"}

    # ---- CNA subset ----
    if cna == "amp":
        df = df[df["copies"] > (2 - et)]
    elif cna == "del":
        df = df[df["copies"] < (2 + et)]
    elif cna == "all":
        pass
    else:
        raise ValueError("cna must be 'amp', 'del', or 'all'")

    if df.empty:
        return {"status": "skipped", "gene": gene, "reason": "no_samples_after_cna_filter"}

    # ---- basic QC ----
    df = df.dropna(subset=list(required))
    if df.empty:
        return {"status": "skipped", "gene": gene, "reason": "all_na_after_dropna"}

    if (df["expr"] < 0).any():
        return {"status": "error", "gene": gene, "reason": "negative_counts"}

    if not df["purity"].between(0, 1).all():
        return {"status": "error", "gene": gene, "reason": "purity_out_of_bounds"}

    if not (df["sf"] > 0).all():
        return {"status": "error", "gene": gene, "reason": "nonpositive_sf"}

    if df["expr"].nunique() < min_unique_counts:
        return {"status": "skipped", "gene": gene, "reason": "too_few_unique_counts"}

    # aneuploid count check
    n_aneup = int((np.abs(df["copies"].astype(float) - 2.0) > (1.0 - et)).sum())
    if n_aneup < min_aneup or (df["expr"] == 0).all():
        return {
            "status": "skipped",
            "gene": gene,
            "n_aneup": n_aneup,
            "reason": "low_aneup_or_all_zero",
        }

    # identifiability check: need some CN deviation mass
    dev_tmp = (df["copies"].astype(float) - 2.0) / 2.0
    if cna == "all" and float(np.abs(dev_tmp).sum()) < min_cn_abs_sum:
        return {
            "status": "skipped",
            "gene": gene,
            "n_aneup": n_aneup,
            "reason": "too_little_cn_variation",
        }

    # ---- subtype encoding -----
    if subtype_order is not None:
        cat = pd.Categorical(df[subtype_col], categories=subtype_order, ordered=True)
        if cat.isna().any():
            bad = df.loc[cat.isna(), subtype_col].unique().tolist()
            return {
                "status": "error",
                "gene": gene,
                "reason": f"unknown_subtypes: {bad}",
            }
        subtype_codes = (cat.codes + 1).astype(int)
        levels = list(cat.categories)
    else:
        levels = sorted(pd.unique(df[subtype_col]).tolist())
        mapping = {lv: i + 1 for i, lv in enumerate(levels)}
        subtype_codes = df[subtype_col].map(mapping).astype(int).to_numpy()

    S = len(levels)
    if S < 2:
        return {
            "status": "skipped",
            "gene": gene,
            "reason": "need_at_least_2_subtypes_present_for_DE",
        }

    # ---- CN covariates (identified) ----
    # effective CN for scaling: treat 0 and 1 as "1 copy"
    CN_eff = df["copies"].astype(float).clip(lower=1.0)
    #df["dose_log"] = np.log(np.maximum(df["copies"].astype(float), 0.1) / 2.0)
    df["dose_log"] = np.log(CN_eff / 2.0)
    df["dev"] = (df["copies"].astype(float) - 2.0) / 2.0

    stan_data = {
        "N": int(len(df)),
        "y": df["expr"].astype(int).to_numpy(),
        "S": int(S),
        "subtype": np.asarray(subtype_codes, dtype=int),
        "sf": df["sf"].to_numpy(dtype=float),
        "purity": df["purity"].to_numpy(dtype=float),
        "dose_log": df["dose_log"].to_numpy(dtype=float),
        "dev": df["dev"].to_numpy(dtype=float),
    }

    # ---- initial values ----
    rng = np.random.default_rng(seed)
    init = {
        "b0_mean": float(rng.normal(0.0, 0.2)),
        "b0_offset": rng.normal(0.0, 0.1, size=S).tolist(),
        "b_scaling_mean": float(rng.normal(0.0, 0.2)),
        "b_scaling_offset": rng.normal(0.0, 0.1, size=S).tolist(),
        "b_dev_mean": float(rng.normal(0.0, 0.05)),
        "b_dev_offset": rng.normal(0.0, 0.05, size=S).tolist(),
        "b_noncancer_log": float(rng.normal(0.0, 0.2)),
        "phi": 1.0,
    }

    fit = model.sample(
        data=stan_data,
        chains=chains,
        iter_warmup=iter_warmup,
        iter_sampling=iter_sampling,
        seed=seed,
        inits=init,
        show_progress=show_progress,
        adapt_delta=adapt_delta,
        max_treedepth=max_treedepth,
    )

    draws = fit.draws_pd()

    # ---- helpers ----
    def q(x: np.ndarray) -> list[float]:
        return np.quantile(x, [0.025, 0.5, 0.975]).tolist()

    def summarize_draw(x: np.ndarray, prefix: str) -> dict:
        qi = q(x)
        return {
            f"{prefix}_mean": float(np.mean(x)),
            f"{prefix}_q025": float(qi[0]),
            f"{prefix}_q50": float(qi[1]),
            f"{prefix}_q975": float(qi[2]),
        }

    # ---- check essential columns ----
    required_cols = [
        "delta_tumor0_log",
        "delta_scaling",
        "delta_dev",
        "phi",
        "b_noncancer_log",
    ]
    missing_cols = [c for c in required_cols if c not in draws.columns]
    if missing_cols:
        return {
            "status": "error",
            "gene": gene,
            "reason": f"missing_draws_columns: {missing_cols}",
        }

    # ---- contrasts ----
    d_tumor = draws["delta_tumor0_log"].to_numpy()
    d_scal = draws["delta_scaling"].to_numpy()
    d_dev = draws["delta_dev"].to_numpy()

    # sign probabilities & lfsr
    p_up_tumor = float((d_tumor > 0).mean())
    lfsr_tumor = float(min(p_up_tumor, 1.0 - p_up_tumor))
    p_rope_tumor = float((np.abs(d_tumor) <= rope_logfc).mean())

    p_up_scal = float((d_scal > 0).mean())
    lfsr_scal = float(min(p_up_scal, 1.0 - p_up_scal))

    p_up_dev = float((d_dev > 0).mean())
    lfsr_dev = float(min(p_up_dev, 1.0 - p_up_dev))

    # log2 fold-change for tumor baseline DE
    ln2 = np.log(2.0)
    lfc_tumor = d_tumor / ln2
    lfc_ci = q(lfc_tumor)

    out: dict[str, object] = {
        "status": "ok",
        "gene": gene,
        "N": int(len(df)),
        "n_aneup": n_aneup,
        "cna": cna,
        "subtype_levels": levels,

        # tumor baseline DE at CN=2 (log2 scale)
        "tumor0_lfc_mean": float(np.mean(lfc_tumor)),
        "tumor0_lfc_q025": float(lfc_ci[0]),
        "tumor0_lfc_q975": float(lfc_ci[2]),
        "p_up_tumor": p_up_tumor,
        "lfsr_tumor": lfsr_tumor,
        "p_rope_tumor": p_rope_tumor,

        # differential CN wiring between subtypes
        "delta_scaling_mean": float(np.mean(d_scal)),
        "delta_scaling_q025": float(q(d_scal)[0]),
        "delta_scaling_q975": float(q(d_scal)[2]),
        "p_up_scaling": p_up_scal,
        "lfsr_scaling": lfsr_scal,

        "delta_dev_mean": float(np.mean(d_dev)),
        "delta_dev_q025": float(q(d_dev)[0]),
        "delta_dev_q975": float(q(d_dev)[2]),
        "p_up_dev": p_up_dev,
        "lfsr_dev": lfsr_dev,

        # dispersion
        "phi_mean": float(np.mean(draws["phi"].to_numpy())),
        "phi_q025": float(q(draws["phi"].to_numpy())[0]),
        "phi_q975": float(q(draws["phi"].to_numpy())[2]),
    }

    # ---- subtype-specific coefficients & dosage summaries -----
    s_iter = range(1, S + 1) if return_all_subtypes else range(1, min(S, 2) + 1)

    for s in s_iter:
        # coefficients b0, b_scaling, b_deviation
        for base in ["b0", "b_scaling", "b_deviation"]:
            col = f"{base}[{s}]"
            if col in draws.columns:
                arr = draws[col].to_numpy()
                out.update(summarize_draw(arr, f"{base}_s{s}"))

        # canonical CN transitions
        col21 = f"lp_2to1[{s}]"
        col23 = f"lp_2to3[{s}]"
        col24 = f"lp_2to4[{s}]"

        if col21 in draws.columns:
            lp21 = draws[col21].to_numpy()
            out.update(summarize_draw(lp21, f"lp_2to1_s{s}"))
            frac21 = np.expm1(lp21)
            out.update(summarize_draw(frac21, f"fracCN_2to1_s{s}"))
            out[f"p_fracCN_2to1_pos_s{s}"] = float((frac21 > eps_frac).mean())
            out[f"p_fracCN_2to1_rope_s{s}"] = float((np.abs(frac21) <= eps_frac).mean())
            out[f"p_fracCN_2to1_neg_s{s}"] = float((frac21 < -eps_frac).mean())

        if col23 in draws.columns:
            lp23 = draws[col23].to_numpy()
            out.update(summarize_draw(lp23, f"lp_2to3_s{s}"))
            frac23 = np.expm1(lp23)
            out.update(summarize_draw(frac23, f"fracCN_2to3_s{s}"))
            out[f"p_fracCN_2to3_pos_s{s}"] = float((frac23 > eps_frac).mean())
            out[f"p_fracCN_2to3_rope_s{s}"] = float((np.abs(frac23) <= eps_frac).mean())
            out[f"p_fracCN_2to3_neg_s{s}"] = float((frac23 < -eps_frac).mean())

        if col24 in draws.columns:
            lp24 = draws[col24].to_numpy()
            out.update(summarize_draw(lp24, f"lp_2to4_s{s}"))
            frac24 = np.expm1(lp24)
            out.update(summarize_draw(frac24, f"fracCN_2to4_s{s}"))
            out[f"p_fracCN_2to4_pos_s{s}"] = float((frac24 > eps_frac).mean())
            out[f"p_fracCN_2to4_rope_s{s}"] = float((np.abs(frac24) <= eps_frac).mean())
            out[f"p_fracCN_2to4_neg_s{s}"] = float((frac24 < -eps_frac).mean())

        # Optional: if you added cancel_index_2to*, p_DC_* in Stan, they will be
        # present in draws and you can either pick them up here or classify later
        # from lp_scaling_* and lp_dev_*.

        # Also summarize lp_scaling_* and lp_dev_* if present (useful for diagnostics)
        for trans in ["2to1", "2to3", "2to4"]:
            for comp in ["scaling", "dev"]:
                col = f"lp_{comp}_{trans}[{s}]"
                if col in draws.columns:
                    arr = draws[col].to_numpy()
                    out.update(summarize_draw(arr, f"lp_{comp}_{trans}_s{s}"))

    # ---- Sampler diagnostics (for model checking) ----
    summ = fit.summary()
    rhat_col = next((c for c in ["R_hat", "Rhat"] if c in summ.columns), None)
    ess_col = next((c for c in ["Ess_bulk", "ESS_bulk", "N_Eff", "Ess"] if c in summ.columns), None)

    # Rhat / ESS for phi 
    if rhat_col and "phi" in summ.index:
        out["Rhat_phi"] = float(summ.loc["phi", rhat_col])
    if ess_col and "phi" in summ.index:
        out["ess_phi"] = float(summ.loc["phi", ess_col])

    # Rhat / ESS over a core set of parameters (for global diagnostics)
    core_params = ["delta_tumor0_log", "delta_scaling", "delta_dev"]
    for s in s_iter:
        core_params += [f"b0[{s}]", f"b_scaling[{s}]", f"b_deviation[{s}]"]

    core_params = [p for p in core_params if p in summ.index]

    if core_params and rhat_col and ess_col:
        rhat_vals = summ.loc[core_params, rhat_col].to_numpy(dtype=float)
        ess_vals = summ.loc[core_params, ess_col].to_numpy(dtype=float)
        out["max_Rhat_core"] = float(np.nanmax(rhat_vals))
        out["min_ESS_core"] = float(np.nanmin(ess_vals))
    else:
        out["max_Rhat_core"] = np.nan
        out["min_ESS_core"] = np.nan

    # Divergences and treedepth from sampler params (if present)
    if "divergent__" in draws.columns:
        out["n_divergent"] = int(draws["divergent__"].sum())
    if "treedepth__" in draws.columns:
        td = draws["treedepth__"].to_numpy()
        out["max_treedepth"] = int(td.max())
        out["n_max_treedepth"] = int((td >= max_treedepth).sum())

    # simple flag based on diagnostics
    out["fit_flag"] = "ok"
    if (
        ("max_Rhat_core" in out and out["max_Rhat_core"] is not np.nan and out["max_Rhat_core"] > 1.05)
        or ("min_ESS_core" in out and out["min_ESS_core"] is not np.nan and out["min_ESS_core"] < 100)
        or ("n_divergent" in out and out["n_divergent"] > 0)
    ):
        out["fit_flag"] = "warn"

    return out

In [13]:
res_singleGene = fit_one_gene_de(gene_df, 
                                 model,
                                 chains=4,
                                 iter_warmup = 500,
                                 iter_sampling = 500,
                                 adapt_delta = 0.9)

10:58:27 - cmdstanpy - INFO - CmdStan start processing
10:58:27 - cmdstanpy - INFO - Chain [1] start processing
10:58:27 - cmdstanpy - INFO - Chain [2] start processing
10:58:27 - cmdstanpy - INFO - Chain [3] start processing
10:58:27 - cmdstanpy - INFO - Chain [4] start processing
10:58:39 - cmdstanpy - INFO - Chain [2] done processing
10:58:41 - cmdstanpy - INFO - Chain [4] done processing
10:58:41 - cmdstanpy - INFO - Chain [3] done processing
11:11:16 - cmdstanpy - INFO - Chain [1] done processing
11:11:16 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: neg_binomial_2_lpmf: Location parameter[1] is inf, but must be positive finite! (in 'differential_dosage_model.stan', line 105, column 2 to column 30)
	Exception: neg_binomial_2_lpmf: Location parameter[1] is inf, but must be positive finite! (in 'differential_dosage_model.stan', line 105, column 2 to column 30)
	Exception: neg_binomial_2_lpmf: Location parameter[1] is inf, but must be positive finite! (in 'diffe

In [15]:
res_singleGene

{'status': 'ok',
 'gene': 'SMAD4',
 'N': 986,
 'n_aneup': 333,
 'cna': 'all',
 'subtype_levels': ['MSI', 'MSS'],
 'tumor0_lfc_mean': -5.254321543965407,
 'tumor0_lfc_q025': -20.503004843097592,
 'tumor0_lfc_q975': -0.06559795130850672,
 'p_up_tumor': 0.0005,
 'lfsr_tumor': 0.0005,
 'p_rope_tumor': 0.705,
 'delta_scaling_mean': 1.208814725277,
 'delta_scaling_q025': -0.633800025,
 'delta_scaling_q975': 5.40672025,
 'p_up_scaling': 0.4105,
 'lfsr_scaling': 0.4105,
 'delta_dev_mean': 0.36425213192234995,
 'delta_dev_q025': -0.023919395,
 'delta_dev_q975': 0.7829721,
 'p_up_dev': 0.964,
 'lfsr_dev': 0.03600000000000003,
 'phi_mean': 11.242201849999999,
 'phi_q025': 10.0005,
 'phi_q975': 12.607024999999998,
 'b0_s1_mean': 7.884915374999999,
 'b0_s1_q025': 7.360229749999999,
 'b0_s1_q50': 8.04332,
 'b0_s1_q975': 8.133584,
 'b_scaling_s1_mean': 1.477640448,
 'b_scaling_s1_q025': 0.56031615,
 'b_scaling_s1_q50': 1.099015,
 'b_scaling_s1_q975': 2.93194025,
 'b_deviation_s1_mean': 0.235514683351

In [15]:
res_df = pd.DataFrame([res_singleGene])
res_df.head()

,status,gene,N,n_aneup,cna,subtype_levels,tumor0_lfc_mean,tumor0_lfc_q025,tumor0_lfc_q975,p_up_tumor,...,p_lp_dev_2to4_neg_s2,cancel_index_2to4_s2_mean,cancel_index_2to4_s2_q025,cancel_index_2to4_s2_q50,cancel_index_2to4_s2_q975,p_DC_gain_s2,p_DC_loss_s2,Rhat_phi,ess_phi,fit_flag
0,ok,SMAD4,986,333,all,"[MSI, MSS]",-5.035997,-19.697115,-0.054649,0.001,...,0.3455,-0.027129,-0.333996,-0.044149,0.183303,0.0,0.0,2.95237,2.24508,warn


#### Parallel run on multiple genes

In [13]:
gene_groups = {g: gdf for g, gdf in gene_df.groupby("gene", sort=False)}

def run_gene(g):
    return fit_one_gene_de(
        gene_groups[g],
        model,
        cna="all",
        chains=4,
        iter_warmup=500,
        iter_sampling=500,
        adapt_delta=0.90
    )

genes = list(gene_groups.keys())

results = Parallel(n_jobs=8, backend="loky")(
    delayed(run_gene)(g) for g in genes
)

15:57:03 - cmdstanpy - INFO - CmdStan start processing
15:57:03 - cmdstanpy - INFO - CmdStan start processing
15:57:03 - cmdstanpy - INFO - CmdStan start processing
15:57:03 - cmdstanpy - INFO - CmdStan start processing
15:57:03 - cmdstanpy - INFO - Chain [1] start processing
15:57:03 - cmdstanpy - INFO - Chain [1] start processing
15:57:03 - cmdstanpy - INFO - Chain [1] start processing
15:57:03 - cmdstanpy - INFO - Chain [2] start processing
15:57:03 - cmdstanpy - INFO - Chain [2] start processing
15:57:03 - cmdstanpy - INFO - Chain [1] start processing
15:57:03 - cmdstanpy - INFO - CmdStan start processing
15:57:03 - cmdstanpy - INFO - Chain [3] start processing
15:57:03 - cmdstanpy - INFO - Chain [2] start processing
15:57:03 - cmdstanpy - INFO - Chain [2] start processing
15:57:03 - cmdstanpy - INFO - Chain [3] start processing
15:57:03 - cmdstanpy - INFO - Chain [3] start processing
15:57:03 - cmdstanpy - INFO - Chain [1] start processing
15:57:03 - cmdstanpy - INFO - Chain [2] s

In [55]:
res_df = pd.DataFrame(results)
res_df.head()

,status,gene,N,n_aneup,cna,subtype_levels,tumor0_lfc_mean,tumor0_lfc_q025,tumor0_lfc_q975,p_up_tumor,...,p_lp_dev_2to4_neg_s2,cancel_index_2to4_s2_mean,cancel_index_2to4_s2_q025,cancel_index_2to4_s2_q50,cancel_index_2to4_s2_q975,p_DC_gain_s2,p_DC_loss_s2,Rhat_phi,ess_phi,fit_flag
0,ok,BRAF,986,139,all,"[MSI, MSS]",0.515110,0.417934,0.612417,1.000,...,0.0730,0.170276,-0.197589,0.128018,0.784141,0.0,0.0,0.999300,2299.740496,ok
1,ok,KRAS,986,69,all,"[MSI, MSS]",-0.015753,-0.122581,0.092460,0.386,...,0.0605,-0.012394,-0.102862,-0.017951,0.106385,0.0,0.0,0.999700,2072.642570,ok
2,ok,TP53,986,217,all,"[MSI, MSS]",-0.018923,-0.176240,0.130874,0.405,...,0.1765,0.063260,-0.351281,0.037346,0.659321,0.0,0.0,2.528062,2.346820,warn


In [ ]:
# Group data by gene
gene_groups = {g: gdf for g, gdf in gene_df.groupby("gene", sort=False)}
genes = list(gene_groups.keys())

def run_gene_safe(g):
    """Run Stan for a single gene."""
    try:
        fit = fit_one_gene_de(
            gene_groups[g],
            model,
            cna="all",
            chains=4,
            iter_warmup=500,
            iter_sampling=500,
            adapt_delta=0.90,
        )
        return {
            "gene": g,
            "success": True,
            "result": fit,
            "error": None,
        }
    except Exception as e:
        # Get full traceback as string (so you can inspect it later)
        tb = traceback.format_exc()
        # This print will go to your Slurm log
        print(f"[run_gene_safe] Gene {g} FAILED with error:\n{tb}", flush=True)
        return {
            "gene": g,
            "success": False,
            "result": None,
            "error": str(e),
        }

results = Parallel(n_jobs=5, backend="loky")(
    delayed(run_gene_safe)(g) for g in genes
)

# Separate good and bad genes
good_results = {r["gene"]: r["result"] for r in results if r["success"]}
failed_genes = [r["gene"] for r in results if not r["success"]]

print(f"[summary] Successful genes: {len(good_results)}", flush=True)
print(f"[summary] Failed genes: {len(failed_genes)}", flush=True)
if failed_genes:
    print(f"[summary] Failed gene IDs: {failed_genes}", flush=True)

In [65]:
results

[{'status': 'ok',
  'gene': 'BRAF',
  'N': 986,
  'n_aneup': 139,
  'cna': 'all',
  'subtype_levels': ['MSI', 'MSS'],
  'tumor0_lfc_mean': 0.5151095784759115,
  'tumor0_lfc_q025': 0.41793421819299575,
  'tumor0_lfc_q975': 0.6124167231800325,
  'p_up_tumor': 1.0,
  'lfsr_tumor': 0.0,
  'p_rope_tumor': 0.0,
  'delta_scaling_mean': -0.37801920144,
  'delta_scaling_q025': -0.7982736500000001,
  'delta_scaling_q975': 0.051960689999999865,
  'p_up_scaling': 0.0425,
  'lfsr_scaling': 0.0425,
  'delta_dev_mean': -0.036865339251849996,
  'delta_dev_q025': -0.298535825,
  'delta_dev_q975': 0.22857424999999998,
  'p_up_dev': 0.389,
  'lfsr_dev': 0.389,
  'phi_mean': 30.84211745,
  'phi_q025': 28.134182499999998,
  'phi_q975': 33.62240249999999,
  'b0_s1_mean': 7.30529091,
  'b0_s1_q025': 7.22689425,
  'b0_s1_q50': 7.30698,
  'b0_s1_q975': 7.384856,
  'b_scaling_s1_mean': 1.2163971339999997,
  'b_scaling_s1_q025': 0.8219335,
  'b_scaling_s1_q50': 1.212635,
  'b_scaling_s1_q975': 1.6027369999999999

In [15]:
# Downstream results interpretation

@dataclass
class InterpretThresholds:
    # DE thresholds 
    de_lfsr_sig: float = 0.05        # call DE if lfsr <= this
    de_rope_high: float = 0.90       # call DE-null if p_rope >= this

    # Rewiring thresholds 
    rewire_lfsr_sig: float = 0.10    # call rewiring if lfsr <= this

    # --- Dosage class thresholds (per subtype) ---
    # We use posterior probabilities that our fit fn already computes.
    # "Sensitive" means CN perturbation yields effect beyond ROPE with high prob.
    dose_prob_sens: float = 0.80     # e.g. p_fracCN_2to3_pos >= 0.90 => sensitive to gain
    dose_prob_ins: float = 0.80      # e.g. p_fracCN_2to3_rope >= 0.90 => insensitive to gain

    # Dosage-compensation thresholds (if you output p_DC_gain_sX / p_DC_loss_sX)
    dc_prob: float = 0.80            # call compensated if >= this

    # If p_DC_* are not present, you can fall back to "cancel_index" if you output it:
    cancel_abs_rope: float = 0.20    # |cancel_index| <= this => strong cancellation (optional)

def _get(res: Dict[str, Any], key: str, default=None):
    return res.get(key, default)

def interpret_gene_result(
    res: Dict[str, Any],
    th: InterpretThresholds = InterpretThresholds(),
    assume_pairwise_s2_vs_s1: bool = True,
) -> Dict[str, Any]:
    """
    Interpret one gene's fitted result dict:
      - DE status (tumor baseline, subtype2 vs subtype1)
      - rewiring status (scaling / deviation)
      - dosage class per subtype (DSG / DIG / DCG) with optional gain/loss flags

    Returns a FLAT dict (good for DataFrame rows).
    """

    out: Dict[str, Any] = {
        "gene": _get(res, "gene"),
        "status": _get(res, "status"),
        "fit_flag": _get(res, "fit_flag", "ok"),
        "N": _get(res, "N"),
        "n_aneup": _get(res, "n_aneup"),
    }


    # 1) DE status (tumor baseline)

    lfsr_tumor = _get(res, "lfsr_tumor", np.nan)
    p_rope_tumor = _get(res, "p_rope_tumor", np.nan)

    if np.isfinite(lfsr_tumor) and lfsr_tumor <= th.de_lfsr_sig:
        de_status = "DE"
    elif np.isfinite(p_rope_tumor) and p_rope_tumor >= th.de_rope_high:
        de_status = "DE-null"
    else:
        de_status = "DE-uncertain"

    out.update({
        "de_status": de_status,
        "lfsr_tumor": lfsr_tumor,
        "p_rope_tumor": p_rope_tumor,
        # keep your effect size if present
        "tumor0_lfc_mean": _get(res, "tumor0_lfc_mean", np.nan),
        "tumor0_lfc_q025": _get(res, "tumor0_lfc_q025", np.nan),
        "tumor0_lfc_q975": _get(res, "tumor0_lfc_q975", np.nan),
    })

  
    # 2) Rewiring status (between subtypes)
    
    lfsr_scaling = _get(res, "lfsr_scaling", np.nan)
    lfsr_dev = _get(res, "lfsr_dev", np.nan)

    scaling_rewired = (np.isfinite(lfsr_scaling) and lfsr_scaling <= th.rewire_lfsr_sig)
    dev_rewired     = (np.isfinite(lfsr_dev) and lfsr_dev <= th.rewire_lfsr_sig)

    if scaling_rewired and dev_rewired:
        rewiring = "rewired:scaling+deviation"
    elif scaling_rewired:
        rewiring = "rewired:scaling"
    elif dev_rewired:
        rewiring = "rewired:deviation"
    else:
        rewiring = "not_rewired"

    out.update({
        "rewiring_status": rewiring,
        "lfsr_scaling": lfsr_scaling,
        "lfsr_dev": lfsr_dev,
        "delta_scaling_mean": _get(res, "delta_scaling_mean", np.nan),
        "delta_scaling_q025": _get(res, "delta_scaling_q025", np.nan),
        "delta_scaling_q975": _get(res, "delta_scaling_q975", np.nan),
        "delta_dev_mean": _get(res, "delta_dev_mean", np.nan),
        "delta_dev_q025": _get(res, "delta_dev_q025", np.nan),
        "delta_dev_q975": _get(res, "delta_dev_q975", np.nan),
    })

    
    # 3) Dosage class per subtype
    
    subtype_levels = _get(res, "subtype_levels", None)  # e.g. ["MSI","MSS"]
    if subtype_levels is None:
        # still proceed with s1,s2,... keys
        subtype_levels = []

    # infer how many subtypes are present from outputs (robust)
    # Prefer explicit list; else scan for p_fracCN_2to3_* keys
    S = len(subtype_levels)
    if S == 0:
        # detect max s index present
        s_candidates = []
        for k in res.keys():
            if k.startswith("p_fracCN_2to3_pos_s"):
                try:
                    s_candidates.append(int(k.split("_s")[-1]))
                except Exception:
                    pass
        S = max(s_candidates) if s_candidates else 2  # fallback

    out["S"] = S
    if subtype_levels:
        out["subtype_levels_str"] = "|".join(map(str, subtype_levels))

    def subtype_name(s: int) -> str:
        if 1 <= s <= len(subtype_levels):
            return str(subtype_levels[s-1])
        return f"s{s}"

    # helper for dosage decision per subtype
    def dosage_class_for_subtype(s: int) -> Dict[str, Any]:
        name = subtype_name(s)

        # Evidence for being sensitive vs insensitive for gain (2->3) and loss (2->1)
        p_gain_pos  = _get(res, f"p_fracCN_2to3_pos_s{s}", np.nan)
        p_gain_rope = _get(res, f"p_fracCN_2to3_rope_s{s}", np.nan)
        p_loss_neg  = _get(res, f"p_fracCN_2to1_neg_s{s}", np.nan)
        p_loss_rope = _get(res, f"p_fracCN_2to1_rope_s{s}", np.nan)

        # Optional DC outputs
        p_dc_gain = _get(res, f"p_DC_gain_s{s}", np.nan)
        p_dc_loss = _get(res, f"p_DC_loss_s{s}", np.nan)

        # Optional cancellation index (if you output it)
        cancel_gain_mean = _get(res, f"cancel_index_2to3_s{s}_mean", np.nan)
        cancel_loss_mean = _get(res, f"cancel_index_2to1_s{s}_mean", np.nan)

        # classify gain/loss directionally (useful diagnostics)
        gain_flag = "unknown"
        if np.isfinite(p_gain_pos) and p_gain_pos >= th.dose_prob_sens:
            gain_flag = "sensitive"
        elif np.isfinite(p_gain_rope) and p_gain_rope >= th.dose_prob_ins:
            gain_flag = "insensitive"

        loss_flag = "unknown"
        if np.isfinite(p_loss_neg) and p_loss_neg >= th.dose_prob_sens:
            loss_flag = "sensitive"
        elif np.isfinite(p_loss_rope) and p_loss_rope >= th.dose_prob_ins:
            loss_flag = "insensitive"

        # primary subtype dosage class
        # Priority: DC > DIS > DS > uncertain
        dc_gain = (np.isfinite(p_dc_gain) and p_dc_gain >= th.dc_prob)
        dc_loss = (np.isfinite(p_dc_loss) and p_dc_loss >= th.dc_prob)

        # Fallback DC using cancellation index if p_DC_* missing
        if (not np.isfinite(p_dc_gain)) and np.isfinite(cancel_gain_mean):
            dc_gain = (abs(cancel_gain_mean) <= th.cancel_abs_rope)
        if (not np.isfinite(p_dc_loss)) and np.isfinite(cancel_loss_mean):
            dc_loss = (abs(cancel_loss_mean) <= th.cancel_abs_rope)

        any_dc = dc_gain or dc_loss

        # DIS if both gain and loss are ROPE-mostly (or at least gain ROPE-mostly and loss ROPE-mostly)
        dis_gain = (np.isfinite(p_gain_rope) and p_gain_rope >= th.dose_prob_ins)
        dis_loss = (np.isfinite(p_loss_rope) and p_loss_rope >= th.dose_prob_ins)
        is_dis = dis_gain and dis_loss

        # DS if strong sensitivity at least on one side (commonly gain), ideally both
        ds_gain = (np.isfinite(p_gain_pos) and p_gain_pos >= th.dose_prob_sens)
        ds_loss = (np.isfinite(p_loss_neg) and p_loss_neg >= th.dose_prob_sens)
        is_ds = ds_gain or ds_loss

        if any_dc:
            cls = "DCG"   # dosage-compensated
        elif is_dis:
            cls = "DIG"  # dosage-insensitive
        elif is_ds:
            cls = "DSG"   # dosage-sensitive
        else:
            cls = "UNC"

        # attach a few useful summary numbers if present
        return {
            f"dosage_class_{name}": cls,
            f"gain_flag_{name}": gain_flag,
            f"loss_flag_{name}": loss_flag,
            f"p_gain_pos_{name}": p_gain_pos,
            f"p_gain_rope_{name}": p_gain_rope,
            f"p_loss_neg_{name}": p_loss_neg,
            f"p_loss_rope_{name}": p_loss_rope,
            f"p_DC_gain_{name}": p_dc_gain,
            f"p_DC_loss_{name}": p_dc_loss,
        }

    for s in range(1, S + 1):
        out.update(dosage_class_for_subtype(s))

    # 4) Optional combined label for easy plotting
    
    # Example: "DE-null | not_rewired | MSI:DS, MSS:DS"
    per_sub = []
    for s in range(1, S + 1):
        name = subtype_name(s)
        per_sub.append(f"{name}:{out.get(f'dosage_class_{name}', 'NA')}")
    out["summary_label"] = f"{de_status} | {rewiring} | " + ",".join(per_sub)

    return out

In [17]:
interpreter = [interpret_gene_result(r) for r in results if r.get("status") == "ok"]
interpreter_df = pd.DataFrame(interpreter)
interpreter_df.head()

,gene,status,fit_flag,N,n_aneup,de_status,lfsr_tumor,p_rope_tumor,tumor0_lfc_mean,tumor0_lfc_q025,...,dosage_class_MSS,gain_flag_MSS,loss_flag_MSS,p_gain_pos_MSS,p_gain_rope_MSS,p_loss_neg_MSS,p_loss_rope_MSS,p_DC_gain_MSS,p_DC_loss_MSS,summary_label
0,APC,ok,ok,986,66,DE,0.000,0.0000,-0.472375,-0.582549,...,DSG,sensitive,sensitive,1.0,0.0,1.0,0.0,0.0,0.0,"DE | not_rewired | MSI:DSG,MSS:DSG"
1,BRAF,ok,ok,986,139,DE,0.000,0.0000,0.515110,0.417934,...,DSG,sensitive,sensitive,1.0,0.0,1.0,0.0,0.0,0.0,"DE | rewired:scaling | MSI:DSG,MSS:DSG"
2,KRAS,ok,ok,986,69,DE-null,0.386,1.0000,-0.015753,-0.122581,...,DSG,sensitive,sensitive,1.0,0.0,1.0,0.0,0.0,0.0,"DE-null | not_rewired | MSI:DSG,MSS:DSG"
3,PIK3CA,ok,ok,986,20,DE,0.000,0.0315,0.372497,0.256616,...,DSG,sensitive,sensitive,1.0,0.0,1.0,0.0,0.0,0.0,"DE | not_rewired | MSI:DSG,MSS:DSG"
4,SMAD4,ok,warn,986,333,DE,0.001,0.7360,-5.035997,-19.697115,...,DSG,sensitive,sensitive,1.0,0.0,1.0,0.0,0.0,0.0,"DE | not_rewired | MSI:DSG,MSS:DSG"
